# Applicant Profile Selection Pipeline

**Author:** Levy Thiga Kariuki  
**Student Number:** G20893080  
**Project:** Confidence-Aware Adaptive XAI for Financial Decision Support  
**Purpose:** This notebook documents the development pipeline behind the Streamlit artefact. It is included for reproducibility, dissertation traceability, and clear separation between model development and the deployed prototype.


## Table of Contents

1. Project setup and dependency imports
2. Load saved model artefacts
3. Reload and encode the German Credit dataset
4. Define human-readable feature/value mappings
5. Build SHAP-style explanation helpers
6. Define static/adaptive explanation helpers
7. Select curated applicant profiles
8. Validate profile explanations
9. Save selected profiles for the Streamlit prototype

## Notebook Notes

- This notebook prepares the small set of representative applicant profiles used in the participant-facing Streamlit app.
- The full German Credit dataset is loaded from the UCI public URL, while the selected profile output is saved into `data/selected_applicant_profiles.csv`.
- The notebook is deterministic enough for development review, but the deployed app reads the saved profile CSV rather than rerunning this pipeline.


## 1. Project setup and dependency imports

Imports the libraries used for data handling, model loading, and local explanation generation. The project path constants keep outputs aligned with the repository structure.


In [ ]:
# ============================================================
# Applicant Profiles Notebook
# Confidence-Aware XAI Prototype Preparation
# ============================================================

import pandas as pd
import numpy as np
import joblib
import shap

print("Libraries loaded.")

# -----------------------------
# Project Paths
# -----------------------------
from pathlib import Path

# Resolve project folders whether the notebook is run from the repository root
# or from inside the notebooks/ directory.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "app.py").exists() else CURRENT_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)


Libraries loaded.


## 2. Model artefact loading

Reloads the saved financial prediction model, encoders, and confidence-model components so profile selection matches the deployed Streamlit prototype.


In [ ]:
# -----------------------------
# Load Saved Models
# -----------------------------

final_model = joblib.load(MODELS_DIR / "xgboost_credit_model.pkl")
label_encoders = joblib.load(MODELS_DIR / "label_encoders.pkl")
confidence_model = joblib.load(MODELS_DIR / "confidence_model.pkl")
confidence_scaler = joblib.load(MODELS_DIR / "confidence_scaler.pkl")
confidence_label_encoder = joblib.load(MODELS_DIR / "confidence_label_encoder.pkl")

print("Saved models loaded successfully.")


Saved models loaded successfully.


## 3. Dataset reconstruction and encoding

Reloads the German Credit dataset from UCI and prepares it in the same feature order used during model training.


In [ ]:
# -----------------------------
# Reload German Credit Dataset
# -----------------------------

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

column_names = [
    "checking_account_status",
    "duration_months",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_since",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "present_residence_since",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
    "target"
]

df = pd.read_csv(url, sep=" ", names=column_names)

df["target"] = df["target"].map({1: 0, 2: 1})

print("Dataset loaded.")
print(df.shape)
df.head()


Dataset loaded.
(1000, 21)


,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,people_liable,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,0
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,1
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,0
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,0
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,1


In [ ]:
# -----------------------------
# Encode Dataset Using Saved Label Encoders
# -----------------------------

model_df = df.copy()

for col, encoder in label_encoders.items():
    model_df[col] = encoder.transform(model_df[col])

X = model_df.drop("target", axis=1)
y = model_df["target"]

print("Dataset encoded.")
print(X.shape)


Dataset encoded.
(1000, 20)


## 4. Human-readable feature mapping

Defines labels and value translations so encoded German Credit fields can be presented clearly to participants.


In [ ]:
# -----------------------------
# Human-Readable Feature and Value Mappings
# -----------------------------

feature_name_map = {
    "checking_account_status": "Checking account status",
    "duration_months": "Loan duration",
    "credit_history": "Credit history",
    "purpose": "Loan purpose",
    "credit_amount": "Credit amount",
    "savings_account": "Savings account status",
    "employment_since": "Employment duration",
    "installment_rate": "Instalment rate",
    "personal_status_sex": "Personal status / sex",
    "other_debtors": "Other debtors or guarantors",
    "present_residence_since": "Years at current residence",
    "property": "Property status",
    "age": "Age",
    "other_installment_plans": "Other instalment plans",
    "housing": "Housing status",
    "existing_credits": "Number of existing credits",
    "job": "Job type",
    "people_liable": "Number of dependants",
    "telephone": "Telephone registered",
    "foreign_worker": "Foreign worker status"
}

DM_TO_GBP_APPROX = 0.75

def dm_to_gbp_label(amount):
    gbp_amount = amount * DM_TO_GBP_APPROX
    return f"approx. £{gbp_amount:,.0f}"

value_maps = {
    "checking_account_status": {
        "A11": "Negative balance (less than approx. £0)",
        "A12": "Low positive balance (approx. £0 to £150)",
        "A13": "Strong positive balance (greater than or equal to approx. £150)",
        "A14": "No checking account"
    },
    "credit_history": {
        "A30": "no credits taken / all credits paid back duly",
        "A31": "all credits at this bank paid back duly",
        "A32": "existing credits paid back duly until now",
        "A33": "delay in paying off in the past",
        "A34": "critical account / other credits existing"
    },
    "purpose": {
        "A40": "car (new)",
        "A41": "car (used)",
        "A42": "furniture/equipment",
        "A43": "radio/television",
        "A44": "domestic appliances",
        "A45": "repairs",
        "A46": "education",
        "A47": "vacation",
        "A48": "retraining",
        "A49": "business",
        "A410": "others"
    },
    "savings_account": {
        "A61": "Low savings (less than approx. £75)",
        "A62": "Moderate savings (approx. £75 to £375)",
        "A63": "High savings (approx. £375 to £750)",
        "A64": "Very high savings (greater than or equal to approx. £750)",
        "A65": "No savings information"
    },
    "employment_since": {
        "A71": "unemployed",
        "A72": "less than 1 year",
        "A73": "1 to 4 years",
        "A74": "4 to 7 years",
        "A75": "greater than 7 years"
    },
    "personal_status_sex": {
        "A91": "male divorced/separated",
        "A92": "female divorced/separated/married",
        "A93": "male single",
        "A94": "male married/widowed",
        "A95": "female single"
    },
    "other_debtors": {
        "A101": "none",
        "A102": "co-applicant",
        "A103": "guarantor"
    },
    "property": {
        "A121": "real estate",
        "A122": "building society savings agreement / life insurance",
        "A123": "car or other property",
        "A124": "unknown / no property"
    },
    "other_installment_plans": {
        "A141": "bank",
        "A142": "stores",
        "A143": "none"
    },
    "housing": {
        "A151": "rent",
        "A152": "own",
        "A153": "for free"
    },
    "job": {
        "A171": "unemployed / unskilled non-resident",
        "A172": "unskilled resident",
        "A173": "skilled employee / official",
        "A174": "management / self-employed / highly qualified"
    },
    "telephone": {
        "A191": "none",
        "A192": "yes, registered"
    },
    "foreign_worker": {
        "A201": "yes",
        "A202": "no"
    }
}

print("Mappings created.")


Mappings created.


In [ ]:
# -----------------------------
# Decode Feature Values
# -----------------------------

def decode_feature_value(feature, encoded_value):
    if feature == "credit_amount":
        return dm_to_gbp_label(float(encoded_value))

    if feature == "duration_months":
        return f"{int(encoded_value)} months"

    if feature == "age":
        return f"{int(encoded_value)} years"

    if feature == "existing_credits":
        return f"{int(encoded_value)} existing credit(s)"

    if feature == "people_liable":
        return f"{int(encoded_value)} dependant(s)"

    if feature == "installment_rate":
        return f"{int(encoded_value)}% of disposable income"

    if feature == "present_residence_since":
        return f"{int(encoded_value)} year(s)"

    if feature in label_encoders:
        original_code = label_encoders[feature].inverse_transform([int(encoded_value)])[0]

        if feature in value_maps:
            return value_maps[feature].get(original_code, original_code)

        return original_code

    return encoded_value


## 5. Local explanation helpers

Builds SHAP-based utilities for ranking applicant-level feature impacts and generating static/adaptive explanation text.


In [ ]:
# -----------------------------
# Create SHAP Explainer
# -----------------------------

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)

print("SHAP values created.")
print(np.array(shap_values).shape)


SHAP values created.
(1000, 20)


In [ ]:
# -----------------------------
# SHAP Explanation Functions
# -----------------------------

def get_human_readable_shap_table(sample_index, X_data, shap_values, top_n=5):
    shap_row = shap_values[sample_index]

    explanation_df = pd.DataFrame({
        "Feature": X_data.columns,
        "Raw Feature Value": X_data.iloc[sample_index].values,
        "SHAP Value": shap_row,
        "Absolute Impact": np.abs(shap_row)
    })

    explanation_df["Feature Name"] = explanation_df["Feature"].map(
        lambda x: feature_name_map.get(x, x)
    )

    explanation_df["Readable Value"] = explanation_df.apply(
        lambda row: decode_feature_value(row["Feature"], row["Raw Feature Value"]),
        axis=1
    )

    explanation_df["Effect on Risk"] = explanation_df["SHAP Value"].apply(
        lambda x: "Increased risk" if x > 0 else "Reduced risk"
    )

    explanation_df = explanation_df.sort_values(
        by="Absolute Impact",
        ascending=False
    )

    return explanation_df[
        ["Feature Name", "Readable Value", "Effect on Risk", "SHAP Value", "Absolute Impact"]
    ].head(top_n)


def generate_plain_english_explanation(sample_index, X_data, shap_values, top_n=5):
    readable_df = get_human_readable_shap_table(
        sample_index=sample_index,
        X_data=X_data,
        shap_values=shap_values,
        top_n=top_n
    )

    explanation_lines = []

    for _, row in readable_df.iterrows():
        effect = "increased" if row["SHAP Value"] > 0 else "reduced"

        explanation_lines.append(
            f"{row['Feature Name']} ({row['Readable Value']}) {effect} the predicted credit risk."
        )

    return explanation_lines


In [ ]:
# -----------------------------
# Static and Adaptive Explanation Functions
# -----------------------------

def generate_static_explanation(sample_index, X_data, shap_values):
    explanation_lines = generate_plain_english_explanation(
        sample_index=sample_index,
        X_data=X_data,
        shap_values=shap_values,
        top_n=5
    )

    return {
        "condition": "Static",
        "confidence_level": "Not used",
        "explanation_depth": "Standard",
        "top_n_features": 5,
        "explanation": explanation_lines
    }


def generate_adaptive_explanation(sample_index, X_data, shap_values, confidence_level):
    confidence_level = confidence_level.lower()

    if confidence_level == "low":
        top_n = 6
        explanation_depth = "Detailed"
        guidance = (
            "Because the system estimated low confidence, it provides a more detailed "
            "explanation with additional contributing factors."
        )

    elif confidence_level == "medium":
        top_n = 4
        explanation_depth = "Moderate"
        guidance = (
            "Because the system estimated moderate confidence, it provides a balanced "
            "explanation with the main contributing factors."
        )

    elif confidence_level == "high":
        top_n = 2
        explanation_depth = "Concise"
        guidance = (
            "Because the system estimated high confidence, it provides a concise "
            "explanation focused on the strongest contributing factors."
        )

    else:
        raise ValueError("confidence_level must be 'low', 'medium', or 'high'.")

    explanation_lines = generate_plain_english_explanation(
        sample_index=sample_index,
        X_data=X_data,
        shap_values=shap_values,
        top_n=top_n
    )

    return {
        "condition": "Adaptive",
        "confidence_level": confidence_level.capitalize(),
        "explanation_depth": explanation_depth,
        "top_n_features": top_n,
        "guidance": guidance,
        "explanation": explanation_lines
    }


## 6. Confidence and adaptation helpers

Defines the behavioural-confidence helper used to prototype confidence-aware explanation depth.


In [ ]:
# -----------------------------
# Confidence Prediction Function
# -----------------------------

def predict_user_confidence(
    decision_time,
    click_count,
    scroll_depth,
    hover_count,
    explanation_view_time
):
    input_df = pd.DataFrame({
        "decision_time": [decision_time],
        "click_count": [click_count],
        "scroll_depth": [scroll_depth],
        "hover_count": [hover_count],
        "explanation_view_time": [explanation_view_time]
    })

    input_scaled = confidence_scaler.transform(input_df)
    prediction_encoded = confidence_model.predict(input_scaled)[0]
    prediction_label = confidence_label_encoder.inverse_transform([prediction_encoded])[0]

    return prediction_label


## 7. Applicant profile selection

Selects representative applicant cases from the model output to support a controlled user-study flow.


In [ ]:
# -----------------------------
# Select Six Predefined Applicant Profiles
# -----------------------------

prediction_probs = final_model.predict_proba(X)[:, 1]
predictions = final_model.predict(X)

profile_candidates = pd.DataFrame({
    "index": X.index,
    "actual_target": y,
    "predicted_target": predictions,
    "probability_bad_credit": prediction_probs
})

# 2 lowest-risk predicted good-credit cases
low_risk_cases = profile_candidates[
    profile_candidates["predicted_target"] == 0
].sort_values(
    "probability_bad_credit",
    ascending=True
).head(2)

# 2 highest-risk predicted bad-credit cases
high_risk_cases = profile_candidates[
    profile_candidates["predicted_target"] == 1
].sort_values(
    "probability_bad_credit",
    ascending=False
).head(2)

# 2 most borderline cases closest to 0.50 probability
borderline_cases = profile_candidates.iloc[
    (profile_candidates["probability_bad_credit"] - 0.5).abs().argsort()
].head(2)

selected_profiles = pd.DataFrame([
    {
        "Profile ID": "Applicant A",
        "Profile Type": "Low-risk example",
        "Dataset Index": int(low_risk_cases.iloc[0]["index"]),
        "Predicted Risk": "Good Credit",
        "Probability Bad Credit": low_risk_cases.iloc[0]["probability_bad_credit"]
    },
    {
        "Profile ID": "Applicant B",
        "Profile Type": "Low-risk example",
        "Dataset Index": int(low_risk_cases.iloc[1]["index"]),
        "Predicted Risk": "Good Credit",
        "Probability Bad Credit": low_risk_cases.iloc[1]["probability_bad_credit"]
    },
    {
        "Profile ID": "Applicant C",
        "Profile Type": "High-risk example",
        "Dataset Index": int(high_risk_cases.iloc[0]["index"]),
        "Predicted Risk": "Bad Credit",
        "Probability Bad Credit": high_risk_cases.iloc[0]["probability_bad_credit"]
    },
    {
        "Profile ID": "Applicant D",
        "Profile Type": "High-risk example",
        "Dataset Index": int(high_risk_cases.iloc[1]["index"]),
        "Predicted Risk": "Bad Credit",
        "Probability Bad Credit": high_risk_cases.iloc[1]["probability_bad_credit"]
    },
    {
        "Profile ID": "Applicant E",
        "Profile Type": "Borderline example",
        "Dataset Index": int(borderline_cases.iloc[0]["index"]),
        "Predicted Risk": "Bad Credit" if borderline_cases.iloc[0]["predicted_target"] == 1 else "Good Credit",
        "Probability Bad Credit": borderline_cases.iloc[0]["probability_bad_credit"]
    },
    {
        "Profile ID": "Applicant F",
        "Profile Type": "Borderline example",
        "Dataset Index": int(borderline_cases.iloc[1]["index"]),
        "Predicted Risk": "Bad Credit" if borderline_cases.iloc[1]["predicted_target"] == 1 else "Good Credit",
        "Probability Bad Credit": borderline_cases.iloc[1]["probability_bad_credit"]
    }
])

selected_profiles


,Profile ID,Profile Type,Dataset Index,Predicted Risk,Probability Bad Credit
0,Applicant A,Low-risk example,654,Good Credit,0.000080
1,Applicant B,Low-risk example,406,Good Credit,0.000105
2,Applicant C,High-risk example,307,Bad Credit,0.998711
3,Applicant D,High-risk example,334,Bad Credit,0.998326
4,Applicant E,Borderline example,844,Good Credit,0.482328
5,Applicant F,Borderline example,477,Bad Credit,0.519360


In [ ]:
# -----------------------------
# Display Applicant Profile
# -----------------------------

def get_applicant_profile(sample_index, X_data):
    row = X_data.iloc[sample_index]

    profile = {
        "Checking account status": decode_feature_value("checking_account_status", row["checking_account_status"]),
        "Savings account status": decode_feature_value("savings_account", row["savings_account"]),
        "Credit history": decode_feature_value("credit_history", row["credit_history"]),
        "Loan purpose": decode_feature_value("purpose", row["purpose"]),
        "Credit amount": decode_feature_value("credit_amount", row["credit_amount"]),
        "Loan duration": decode_feature_value("duration_months", row["duration_months"]),
        "Employment duration": decode_feature_value("employment_since", row["employment_since"]),
        "Age": decode_feature_value("age", row["age"]),
        "Housing status": decode_feature_value("housing", row["housing"]),
        "Property status": decode_feature_value("property", row["property"]),
        "Job type": decode_feature_value("job", row["job"])
    }

    return profile


def display_applicant_profile(sample_index, X_data):
    profile = get_applicant_profile(sample_index, X_data)

    print("=" * 60)
    print(f"Applicant Profile - Dataset Index {sample_index}")
    print("=" * 60)

    for key, value in profile.items():
        print(f"{key}: {value}")


## 8. Validation and export

Checks the selected cases, tests explanation output, and saves the curated profiles into the project data folder.


In [ ]:
# -----------------------------
# Test Applicant Profiles
# -----------------------------

for _, profile in selected_profiles.iterrows():
    sample_index = int(profile["Dataset Index"])

    print("\n\n")
    display_applicant_profile(sample_index, X)

    prediction = final_model.predict(X.iloc[[sample_index]])[0]
    probability_bad = final_model.predict_proba(X.iloc[[sample_index]])[0][1]

    print("\nAI Prediction:", "Bad Credit" if prediction == 1 else "Good Credit")
    print("Probability of Bad Credit:", round(probability_bad, 4))

    explanation = generate_static_explanation(
        sample_index=sample_index,
        X_data=X,
        shap_values=shap_values
    )

    print("\nStatic Explanation:")
    for line in explanation["explanation"]:
        print("-", line)





Applicant Profile - Dataset Index 654
Checking account status: No checking account
Savings account status: Low savings (less than approx. £75)
Credit history: critical account / other credits existing
Loan purpose: car (used)
Credit amount: approx. £1,760
Loan duration: 24 months
Employment duration: 4 to 7 years
Age: 35 years
Housing status: own
Property status: car or other property
Job type: skilled employee / official

AI Prediction: Good Credit
Probability of Bad Credit: 1e-04

Static Explanation:
- Checking account status (No checking account) reduced the predicted credit risk.
- Credit history (critical account / other credits existing) reduced the predicted credit risk.
- Loan purpose (car (used)) reduced the predicted credit risk.
- Credit amount (approx. £1,760) reduced the predicted credit risk.
- Employment duration (4 to 7 years) reduced the predicted credit risk.



Applicant Profile - Dataset Index 406
Checking account status: No checking account
Savings account statu

In [ ]:
# -----------------------------
# Test Adaptive Explanation on One Profile
# -----------------------------

sample_index = int(selected_profiles.iloc[0]["Dataset Index"])

predicted_confidence = predict_user_confidence(
    decision_time=42,
    click_count=7,
    scroll_depth=0.8,
    hover_count=5,
    explanation_view_time=40
)

adaptive_output = generate_adaptive_explanation(
    sample_index=sample_index,
    X_data=X,
    shap_values=shap_values,
    confidence_level=predicted_confidence
)

print("Predicted Confidence:", predicted_confidence)
print("Explanation Depth:", adaptive_output["explanation_depth"])
print("Top Features Shown:", adaptive_output["top_n_features"])
print(adaptive_output["guidance"])

for line in adaptive_output["explanation"]:
    print("-", line)


Predicted Confidence: medium
Explanation Depth: Moderate
Top Features Shown: 4
Because the system estimated moderate confidence, it provides a balanced explanation with the main contributing factors.
- Checking account status (No checking account) reduced the predicted credit risk.
- Credit history (critical account / other credits existing) reduced the predicted credit risk.
- Loan purpose (car (used)) reduced the predicted credit risk.
- Credit amount (approx. £1,760) reduced the predicted credit risk.


In [ ]:
# -----------------------------
# Save Selected Applicant Profiles
# -----------------------------

selected_profiles.to_csv(DATA_DIR / "selected_applicant_profiles.csv", index=False)

print("Selected applicant profiles saved.")


Selected applicant profiles saved.
